# Heart Risk

Studied data set : https://www.kaggle.com/competitions/sdd-hackaton-25-heart-risk/data

In [1]:
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import plotly.express as px

In [2]:
# df_train = pd.read_csv('./data/train.csv')
# df_extra = pd.read_csv('./data/extra.csv')

In [38]:
# df_full = pd.concat([df_train.sample(frac=1), df_extra.sample(frac=0.5)])
df_full = pd.concat([pd.read_csv('./data/train.csv').sample(frac=0.5),  pd.read_csv('./data/extra.csv').sample(frac=0.5)])

# del df_train
# del df_extra

In [39]:
[col for col in df_full.columns if 'CVD' in col.upper()]

['CVDSTRK3']

In [4]:
df_full.head()

,_STATE,FMONTH,IDATE,IMONTH,IDAY,IYEAR,DISPCODE,SEQNO,_PSU,CTELENM1,...,DRNKANY6,DROCDY4_,_RFBING6,_DRNKWK2,_RFDRHV8,_FLSHOT7,_PNEUMO3,_AIDTST4,ID,TARGET
57533,27.0,5.0,5232022,5,23,2022,1100.0,2022000593,2.022001e+09,1.0,...,2.0,5.397605e-79,1.0,5.397605e-79,1.0,9.0,9.0,2.0,57533,False
166085,23.0,4.0,4022022,4,2,2022,1100.0,2022002957,2.022003e+09,1.0,...,2.0,5.397605e-79,1.0,5.397605e-79,1.0,1.0,2.0,9.0,166085,True
61644,8.0,9.0,9122022,9,12,2022,1100.0,2022007738,2.022008e+09,NaN,...,1.0,1.700000e+01,1.0,3.500000e+02,1.0,2.0,1.0,1.0,61644,False
84844,25.0,1.0,1172022,1,17,2022,1100.0,2022005081,2.022005e+09,NaN,...,2.0,5.397605e-79,1.0,5.397605e-79,1.0,NaN,NaN,1.0,84844,False
197268,36.0,12.0,12212022,12,21,2022,1100.0,2022017502,2.022018e+09,NaN,...,1.0,3.000000e+00,1.0,4.700000e+01,1.0,NaN,NaN,2.0,197268,False


Ce dataset commence par une trentaine de colonnes dédiées aux données de l'appel. Il comporte ensuite une majorité de données de santé de différents types: 
- Des données démographiques telles que l'âge, la taille ou le poids ou l'emploi.
- Des données liées au déficit physique et mental du sujet
- Des données liées au tabagisme, à la prise d'alcool ou de drogue
- Des données liées au cancer 
- Des données liées aux vaccins
- Des données liées au Covid
- Des données liées au diabète
- Des données liées aux grossesses

Enfin le dataset se termine par un certain nombre de données calculées à partir des autres colonnes. 


### Heart risk geographic repartition

In [5]:
data_map = df_full.groupby('_STATE').agg(
    TARGET_MEAN=('TARGET', 'mean'),
    COUNT=('TARGET', 'count')
).reset_index()

dict_idx_to_state_str = {
    1: 'AL', 2: 'AK', 4: 'AZ', 5: 'AR', 6: 'CA', 8: 'CO', 9: 'CT', 10: 'DE',
    11: 'DC', 12: 'FL', 13: 'GA', 15: 'HI', 16: 'ID', 17: 'IL', 18: 'IN', 19: 'IA',
    20: 'KS', 21: 'KY', 22: 'LA', 23: 'ME', 24: 'MD', 25: 'MA', 26: 'MI', 27: 'MN',
    28: 'MS', 29: 'MO', 30: 'MT', 31: 'NE', 32: 'NV', 33: 'NH', 34: 'NJ', 35: 'NM',
    36: 'NY', 37: 'NC', 38: 'ND', 39: 'OH', 40: 'OK', 41: 'OR', 42: 'PA', 44: 'RI',
    45: 'SC', 46: 'SD', 47: 'TN', 48: 'TX', 49: 'UT', 50: 'VT', 51: 'VA', 53: 'WA',
    54: 'WV', 55: 'WI', 56: 'WY', 66: 'GU', 72: 'PR', 78: 'VI'
}

In [6]:
fig = go.Figure(data=go.Choropleth(
    locations=data_map.index.map(dict_idx_to_state_str),
    z=data_map['TARGET_MEAN'],
    locationmode='USA-states',
    colorscale='Reds',
    colorbar=dict(
        title="Target mean",
        tickformat=".0%",
    ),
    text=data_map['COUNT'],
    hovertemplate=(
        "<b>%{location}</b><br>"         # Nom de l'état
        "Target: %{z:.1%}<br>"           # TARGET_MEAN en %
        "Samples: %{text:.0f}<extra></extra>"  # COUNT
    )
))

fig.update_layout(
    title_text='Heart risk per US state',
    geo=dict(
        scope='usa',
        projection=go.layout.geo.Projection(type='albers usa'),
        showlakes=True,
        lakecolor='rgb(255, 255, 255)'
    )
)
fig.show()

La carte montre le pourcentage de personnes ayant eu un problème cardiaque par état. Par exemple, 14% des personnes interviewées habitant au Texas ont déjà eu un problème cardiaque. 

### Age VS Heart disease

In [7]:
# Suppose df_full est ton DataFrame
df = df_full.copy()

# Crée des labels d'âge lisibles
age_labels = {
    1: '18-24', 2: '25-29', 3: '30-34', 4: '35-39',
    5: '40-44', 6: '45-49', 7: '50-54', 8: '55-59',
    9: '60-64', 10: '65-69', 11: '70-74', 12: '75-79',
    13: '80+', 14: 'Missing'
}
df['AGE_GROUP'] = df['_AGEG5YR'].map(age_labels)
df['SEX_LABEL'] = df['_SEX'].map({1:'M', 2:'F'})

data_agg = df.groupby(['AGE_GROUP','SEX_LABEL','TARGET']).size().reset_index(name='count')

fig = go.Figure()
for sex, sign in [('M', -1), ('F', 1)]:
    df_sex = data_agg[data_agg['SEX_LABEL']==sex]
    for target in [0,1]:
        df_target = df_sex[df_sex['TARGET']==target]
        fig.add_trace(go.Bar(
            y=df_target['AGE_GROUP'],
            x=sign * df_target['count'],   # hommes négatifs pour pyramide inversée
            orientation='h',
            name=f"{sex} TARGET={target}",
            marker_color='lightgrey' if target==0 else 'crimson',
            hovertemplate=(
                "%{y}<br>"
                "Sex: "+sex+"<br>"
                "TARGET: "+str(target)+"<br>"
                "Count: %{x}<extra></extra>"
            )
        ))

fig.update_layout(
    title="Age repartition of heart risk",
    barmode='overlay',
    bargap=0.1,
    xaxis=dict(
        title="Number of individuals",
        tickvals=[-100000,-50000,0,50000,100000],  # Ajuster selon tes données
        ticktext=[100000,50000,0,50000,100000]
    ),
    yaxis=dict(title="Age group"),
    template="plotly_white"
)

fig.show()

Les barres de gauche représentent les hommes, celles de droite les femmes. Elles comptent le nombres de personnes ayant eu un problème cardiaque (en rouge) et le nombre n'en ayant jamais eu (en gris) par tranches d'âge. Au global, on remarque non seulement que le risque augmente avec l'âge, mais également que les hommes sont plus touchés. 

### BMI

_BMI5

In [8]:
import plotly.express as px

df_full['BMI'] = df_full['_BMI5'] / 100

fig = px.box(
    df_full,
    x='TARGET',
    y='BMI',
    color='TARGET',
    color_discrete_map={0: 'lightgrey', 1: 'crimson'},
    points=False,  # ou 'outliers' ou False
    title='Distribution of BMI by TARGET',
    labels={'BMI': 'BMI (kg/m²)', 'TARGET': 'Heart risk'},
    width=600

)

fig.show()

Contrairement à ce que l'on pourrait penser, l'indice de masse corporelle ne varie que très peu entre les personnes à risque et celle sans risque. Cela peut être du au défaut de ce data-set qui ne comporte pas de données issues de sujets décédés de problèmes cardiaques. 

### Years smoked

_YRSSMOK

In [ ]:
data_counts = (
    df_full
    .groupby(['_YRSSMOK', 'TARGET'])
    .size()
    .reset_index(name='count')
)

data_counts['total'] = data_counts.groupby('_YRSSMOK')['count'].transform('sum')
data_counts['proportion'] = data_counts['count'] / data_counts['total']

data_0 = data_counts[data_counts['TARGET'] == 0]
data_1 = data_counts[data_counts['TARGET'] == 1]

fig = make_subplots(rows=2,cols=1,
                    shared_xaxes = True)

fig.add_trace(go.Bar(
    x=data_1['_YRSSMOK'],
    y=data_1['total'],
    name='TARGET = 1',
    marker_color='crimson',
    customdata=data_1['count'],
    hovertemplate=(
        "Years Smoking: %{x}<br>"
        "TARGET = 1<br>"
        "Total: %{y:.1%}<br>"
        "Count: %{customdata}<extra></extra>"
    ),
), row=1,col=1)

fig.add_trace(go.Bar(
    x=data_0['_YRSSMOK'],
    y=data_0['total'],
    name='TARGET = 0',
    marker_color='lightgrey',
    customdata=data_0['count'],
    hovertemplate=(
        "Years Smoking: %{x}<br>"
        "TARGET = 0<br>"
        "Total: %{y:.1%}<br>"
        "Count: %{customdata}<extra></extra>"
    ),
), row=1,col=1)

fig.add_trace(go.Bar(
    x=data_1['_YRSSMOK'],
    y=data_1['proportion'],
    name='TARGET = 1',
    marker_color='crimson',
    customdata=data_1['count'],
    hovertemplate=(
        "Years Smoking: %{x}<br>"
        "TARGET = 1<br>"
        "Proportion: %{y:.1%}<br>"
        "Count: %{customdata}<extra></extra>"
    ),
), row=2,col=1)

fig.add_trace(go.Bar(
    x=data_0['_YRSSMOK'],
    y=data_0['proportion'],
    name='TARGET = 0',
    marker_color='lightgrey',
    customdata=data_0['count'],
    hovertemplate=(
        "Years Smoking: %{x}<br>"
        "TARGET = 0<br>"
        "Proportion: %{y:.1%}<br>"
        "Count: %{customdata}<extra></extra>"
    ),
), row=2,col=1)

fig.update_layout(
    title="Distribution of heart risk (TARGET) with years smoking (_YRSSMOK)",
    xaxis_title="Years Smoking (_YRSSMOK)",
    yaxis1_title="Total",
    yaxis2_title="Proportion",
    barmode='stack',
    yaxis2=dict(tickformat=".0%", range=[0, 1.1]),
    template='plotly_white',
    legend_title_text='TARGET',
    showlegend=False,
)

fig.show()

La partie inférieure de ce graphe montre que la proportion de personnes à risque cardiaque augmente avec le nombre d'années pendant laquelle une personne a fumé. Par exemple, la proportion de personnes à risque après un an de consommation de tabac est de 7?4%, contre 21,6% après 50 ans.

### Regular physical activity

EXERANY2

In [10]:
df_full['EXERANY2'] = df_full['EXERANY2'].replace(1,'Active').replace(2,'Sedentary').replace([7,9],'No data')

data_counts = (
    df_full
    .groupby(['EXERANY2', 'TARGET'])
    .size()
    .reset_index(name='count')
)

data_counts['EXERANY2'] = pd.Categorical(
    data_counts['EXERANY2'],
    categories=['Active', 'Sedentary', 'No data'],
    ordered=True
)

data_counts = data_counts.sort_values('EXERANY2').reset_index(drop=True)


data_counts['total'] = data_counts.groupby('EXERANY2')['count'].transform('sum')
data_counts['proportion'] = data_counts['count'] / data_counts['total']


data_0 = data_counts[data_counts['TARGET'] == 0]
data_1 = data_counts[data_counts['TARGET'] == 1]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=data_1['EXERANY2'],
    y=data_1['proportion'],
    name='TARGET = 1',
    marker_color='crimson',
    customdata=data_1['count'],
    hovertemplate=(
        "EXERANY2: %{x}<br>"
        "TARGET = 1<br>"
        "Proportion: %{y:.1%}<br>"
        "Count: %{customdata}<extra></extra>"
    ),
))

fig.add_trace(go.Bar(
    x=data_0['EXERANY2'],
    y=data_0['proportion'],
    name='TARGET = 0',
    marker_color='lightgrey',
    customdata=data_0['count'],
    hovertemplate=(
        "EXERANY2: %{x}<br>"
        "TARGET = 0<br>"
        "Proportion: %{y:.1%}<br>"
        "Count: %{customdata}<extra></extra>"
    ),
))

totals = data_counts[['EXERANY2', 'total']].drop_duplicates()

for _, row in totals.iterrows():
    fig.add_annotation(
        x=row['EXERANY2'],
        y=1.02,  # juste au-dessus de la barre
        text=f"{row['total']:,}",
        showarrow=False,
        font=dict(size=10, color='black')
    )

fig.update_layout(
    title="Proportion of heart risk (TARGET) by physical activity in the last month",
    xaxis_title="EXERANY2",
    yaxis_title="Proportion",
    barmode='stack',
    yaxis=dict(tickformat=".0%", range=[0, 1.1]),
    template='plotly_white',
    legend_title_text='TARGET'
)

fig.show()

/tmp/ipykernel_7184/3331079178.py:19: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Le graphe affiche la proportion de personnes à risque selon leur niveau d'activité physique. On observe un risque plus élevé chez les personnes sédentaires. 

In [13]:
import plotly.graph_objects as go
import pandas as pd
df = df_full.copy()
# --- Nettoyage des données ---
df_sleep = df[(df['SLEPTIM1'].notna()) & (df['SLEPTIM1'] > 0) & (df['SLEPTIM1'] <= 24)]

# --- Agrégation ---
data_agg = (
    df_sleep.groupby(['SLEPTIM1', 'TARGET'])
    .size()
    .reset_index(name='count')
)

# --- Filtrer les heures où moins de 80 personnes ont répondu ---
total_par_heure = data_agg.groupby('SLEPTIM1')['count'].sum()
heures_valide = total_par_heure[total_par_heure >= 80].index
data_agg = data_agg[data_agg['SLEPTIM1'].isin(heures_valide)]

# Recalcul du pourcentage après filtrage
total_par_heure = data_agg.groupby('SLEPTIM1')['count'].transform('sum')
data_agg['percent'] = 100 * data_agg['count'] / total_par_heure

# --- Couleurs ---
colors = {0: "lightgrey", 1: "crimson"}

# --- Figure ---
fig = go.Figure()

# TARGET=1 en bas
for target in [1, 0]:
    df_target = data_agg[data_agg['TARGET'] == target]
    fig.add_trace(go.Bar(
        x=df_target['SLEPTIM1'],
        y=df_target['percent'],
        name=f"TARGET={target}",
        marker_color=colors[target],
        hovertemplate=(
            "Heures de sommeil: %{x}<br>"
            "TARGET: "+str(target)+"<br>"
            "Proportion: %{y:.1f}%<extra></extra>"
        )
    ))

# --- Mise en forme ---
fig.update_layout(
    title="Proportion de personnes avec/sans problème cardiaque selon le nombre d’heures de sommeil",
    xaxis=dict(
        title="Heures de sommeil (SLEPTIM1)",
        tickmode='linear',
        dtick=1
    ),
    yaxis=dict(
        title="Proportion (%)",
        range=[0, 100]
    ),
    barmode='stack',
    bargap=0.1,
    template="plotly_white",
    font=dict(family="Arial", size=12),
    legend_title_text="Problème cardiaque (TARGET)"
)

fig.show()


Le graphe montre la proportion de personnes à risque selon le nombre d'heures de sommeil. On observe qu'entre 6 et 9h de sommeil, le risque est le plus faible. Moins ou plus d'heures de sommeil que cela augmente le risque.

In [23]:
import pandas as pd
import plotly.express as px

# Copier le DataFrame
df_urb = df_full.copy()

# Créer des labels
df_urb['Zone'] = df_urb['_URBSTAT'].map({1:'Urbain', 2:'Rural'})

# Filtrer les valeurs manquantes
df_urb = df_urb[df_urb['Zone'].notna()]

# Calculer la proportion de cardiaques par zone
risk_by_zone = (
    df_urb.groupby('Zone')['TARGET']
    .mean()  # moyenne = proportion TARGET=1
    .reset_index()
)
risk_by_zone['Risk (%)'] = risk_by_zone['TARGET'] * 100

# --- Bar chart ---
fig = px.bar(
    risk_by_zone,
    x='Zone',
    y='Risk (%)',
    text='Risk (%)',
    color='Risk (%)',
    color_continuous_scale='Reds'
)

fig.update_layout(
    title="Proportion de risque cardiaque : Urbain vs Rural",
    xaxis_title="Zone",
    yaxis_title="Proportion de cardiaques (%)",
    template="plotly_white",
    font=dict(family="Arial", size=12)
)

fig.show()


On compare la proportion de personnes à risque selon la zone dans laquelle ils vivent: urbaine ou rurale.

In [32]:
import plotly.express as px
df = df_full.copy()
# Copier le DataFrame et filtrer les valeurs valides
df_health = df[['PHYSHLTH', 'TARGET']].dropna()
df_health = df_health[df_health['PHYSHLTH'].between(1, 30)]  # garder 1-30

# Calculer le total par nombre de jours pour mettre en proportion
df_counts = (
    df_health.groupby(['PHYSHLTH', 'TARGET'])
    .size()
    .reset_index(name='count')
)

# Calcul du total par PHYSHLTH
totals = df_counts.groupby('PHYSHLTH')['count'].transform('sum')
df_counts['proportion'] = df_counts['count'] / totals

# Histogramme en proportion
fig = px.bar(
    df_counts,
    x='PHYSHLTH',
    y='proportion',
    color='TARGET',
    barmode='stack',  # empilé pour faire 100%
    color_discrete_map={0:'lightgrey', 1:'crimson'},
    labels={'PHYSHLTH':'Jours santé physique mauvaise (1-30j)',
            'proportion':'Proportion',
            'TARGET':'Problème cardiaque'}
)


fig.update_layout(
    title="Distribution proportionnelle des jours de mauvaise santé physique selon TARGET",
    yaxis=dict(tickformat=".0%"),  # afficher en pourcentage
    template='plotly_white',
    font=dict(family="Arial", size=12)
)

fig.show()


On observe la proportion de personnes à risque se sentant en mauvaise santé physique un certain nombre de jours par mois. Par exemple, 7% des personnes se sentant mal physiquement 2j par mois sont à risque cardiaque, tandis que 31% des personnes se sentant mal physiquement 24j par mois sont à risque cardiaque.

In [41]:
import pandas as pd
import plotly.express as px

# --- Préparation des données ---
df_heat = df[['_SMOKER3', 'EXERANY2', 'TARGET']].copy()

# Garder seulement les valeurs valides
df_heat = df_heat[
    df_heat['_SMOKER3'].isin([1, 2, 3, 4]) &
    df_heat['EXERANY2'].isin([1, 2])
]

# Mapping plus clair
smoker_labels = {
    4: 'Non-fumeur',
    2: 'Fumeur occasionnel',
    1: 'Fumeur quotidien',
    3: 'Ancien fumeur'
}
exercise_labels = {
    1: 'Actif physiquement',
    2: 'Inactif physiquement'
}

df_heat['_SMOKER3'] = df_heat['_SMOKER3'].map(smoker_labels)
df_heat['EXERANY2'] = df_heat['EXERANY2'].map(exercise_labels)

# Forcer l'ordre sur l'axe X
order_smoker = ['Non-fumeur', 'Fumeur occasionnel', 'Fumeur quotidien', 'Ancien fumeur']
df_heat['_SMOKER3'] = pd.Categorical(df_heat['_SMOKER3'], categories=order_smoker, ordered=True)

# Calcul de la proportion de TARGET=1
df_prop = (
    df_heat.groupby(['_SMOKER3', 'EXERANY2'])['TARGET']
    .mean()
    .reset_index(name='Proportion_cardiaque')
)

# --- Création du graphique ---
fig = px.density_heatmap(
    df_prop,
    x='_SMOKER3',
    y='EXERANY2',
    z='Proportion_cardiaque',
    color_continuous_scale=["#ffe6e6", "#ff9999", "#cc0000", "#660000"],  # palette rouge soutenue
    text_auto='.0%',
    labels={
        '_SMOKER3': 'Statut tabagique',
        'EXERANY2': 'Activité physique',
        'Proportion_cardiaque': 'Proportion cardiaque'
    }
)

fig.update_layout(
    title="Risque cardiaque selon tabagisme et activité physique",
    template='plotly_white',
    font=dict(family="Arial", size=12),
    coloraxis_colorbar=dict(
        title="Proportion de cardiaques",
        tickformat=".0%"
    )
)

fig.show()


/var/folders/gd/kf1g0d9s227gqylz3d572v2m0000gn/T/ipykernel_9898/358970034.py:34: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Ce graphe combine les données de tabagisme et celles d'activité physique pour montrer le pourcentage de personne à risque dans ces catégories combinées. Ainsi 19% des personnes à risque sont d'anciens fumeurs ne pratiquant pas d'activité physique. 

In [42]:
import pandas as pd
import plotly.express as px

# --- Calcul de la proportion globale ---
prop_target = df['TARGET'].mean()

# Créer un petit DataFrame pour le graphique
df_prop = pd.DataFrame({
    'Statut': ['Cardiaque', 'Non-cardiaque'],
    'Proportion': [prop_target, 1 - prop_target]
})

# --- Création du graphique ---
fig = px.bar(
    df_prop,
    x='Statut',
    y='Proportion',
    text='Proportion',
    color='Statut',
    color_discrete_map={
        'Cardiaque': 'crimson',
        'Non-cardiaque': 'lightgrey'
    },
    labels={'Proportion': 'Proportion dans la population totale'}
)

fig.update_traces(
    texttemplate='%{y:.1%}',
    textposition='outside'
)

fig.update_layout(
    title="Proportion de personnes ayant eu un problème cardiaque (TARGET = 1)",
    yaxis=dict(tickformat=".0%"),
    template='plotly_white',
    font=dict(family="Arial", size=12),
    showlegend=False
)

fig.show()
